# Assignment 1: Lumen Beauty and BakeBox

Part 1 follows the Week 2 notebook. Part 2 follows the Week 3 notebook.
Run the whole file from the top (Kernel → Restart & Run All). The CSV files sit in the same folder.

**Assumptions I am using**

- GMV is the value of the goods sold. It is not profit, and it is not cash the founder keeps.
- Streaming costs 800 CNY per hour. That covers the host, the venue, and samples. It does not cover the cost of the products that were sold.
- Growth from week 1 to week 4 means week-4 GMV divided by week-1 GMV, as in the Week 2 TRY IT cell.
- For BakeBox, one basket is one order. `min_support = 0.05`. I do not filter on confidence. Rules are ranked by lift.


## 0 · Setup


In [1]:
import warnings
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 40)
print("pandas", pd.__version__)


pandas 3.0.6


## Part 1 · Lumen Beauty, September 2026

The session file has the same problems as in class: one duplicate row, one watch time stored in seconds, and one missing watch time. I fix those before any ranking. The three tasks below do not use watch time, but the duplicate would inflate the streaming hours and the stream GMV, so it has to go.


In [2]:
sessions = pd.read_csv("livestream_sessions.csv")
print("rows in the export:", sessions.shape[0])
print("duplicate rows:")
print(sessions[sessions.duplicated(keep=False)][["session_id", "date", "gmv"]].to_string(index=False))

df = sessions.drop_duplicates().reset_index(drop=True)
print("sessions after dropping the duplicate:", len(df))

# S09's average watch is longer than the stream. The export stored seconds.
bad = df["avg_watch_min"] > df["duration_min"]
df.loc[bad, "avg_watch_min"] = df.loc[bad, "avg_watch_min"] / 60
print("S09 watch time, in minutes:")
print(df.loc[df["session_id"] == "S09", ["session_id", "duration_min", "avg_watch_min"]].to_string(index=False))
print("missing watch time (left missing on purpose):", int(df["avg_watch_min"].isna().sum()))


rows in the export: 13
duplicate rows:
session_id       date     gmv
       S07 2026-09-16 4409.22
       S07 2026-09-16 4409.22
sessions after dropping the duplicate: 12
S09 watch time, in minutes:
session_id  duration_min  avg_watch_min
       S09           150            4.2
missing watch time (left missing on purpose): 1


## Task 1.1 · Two ways to rank a stream

GMV per viewer = session GMV / peak viewers. Peak viewers is the size of the room. GMV per viewer is how hard that room was sold to. I keep both numbers in the same table so the ratio is not read alone.


In [3]:
df["gmv_per_viewer"] = df["gmv"] / df["viewers_peak"]

by_ratio = df.sort_values("gmv_per_viewer", ascending=False).reset_index(drop=True)
print("Ranked by GMV per viewer")
print(by_ratio[["session_id", "viewers_peak", "gmv", "gmv_per_viewer"]].round(2).to_string(index=False))

by_viewers = df.sort_values("viewers_peak", ascending=False).reset_index(drop=True)
print()
print("Ranked by peak viewers")
print(by_viewers[["session_id", "viewers_peak", "gmv", "gmv_per_viewer"]].round(2).to_string(index=False))

top_ratio = by_ratio.iloc[0]
top_viewers = by_viewers.iloc[0]
print()
print(f"Top by GMV per viewer: {top_ratio['session_id']} at {top_ratio['gmv_per_viewer']:.2f} CNY")
print(f"  peak viewers {int(top_ratio['viewers_peak'])}, session GMV {top_ratio['gmv']:.2f} CNY")
print(f"Top by peak viewers: {top_viewers['session_id']} with {int(top_viewers['viewers_peak'])} viewers")
print(f"  GMV per viewer {top_viewers['gmv_per_viewer']:.2f} CNY, session GMV {top_viewers['gmv']:.2f} CNY")
print("Same session?", top_ratio["session_id"] == top_viewers["session_id"])

# S11 is the giveaway night from class. Comments are high. Sales are not.
s11 = df.loc[df["session_id"] == "S11"].iloc[0]
print(f"S11 giveaway check: {int(s11['comments'])} comments, "
      f"{int(s11['viewers_peak'])} peak viewers, "
      f"{s11['gmv_per_viewer']:.2f} CNY per viewer")


Ranked by GMV per viewer
session_id  viewers_peak      gmv  gmv_per_viewer
       S05           379  6852.76           18.08
       S12          1427 25728.68           18.03
       S09          1305 22936.99           17.58
       S03           844 14783.91           17.52
       S01          1419 22832.04           16.09
       S06          1331 21178.30           15.91
       S04          1249 19137.11           15.32
       S08          1135 17245.42           15.19
       S10           523  7220.19           13.81
       S02          1207 16496.93           13.67
       S07           346  4409.22           12.74
       S11           344  4085.66           11.88

Ranked by peak viewers
session_id  viewers_peak      gmv  gmv_per_viewer
       S12          1427 25728.68           18.03
       S01          1419 22832.04           16.09
       S06          1331 21178.30           15.91
       S09          1305 22936.99           17.58
       S04          1249 19137.11           15.32
 

**Reading.** S05 leads on GMV per viewer, at 18.08 CNY. It is not the biggest room. S12 has the most peak viewers, 1,427, and its GMV per viewer is 18.03 CNY. The two winners are not the same session.

Peak viewers answers "how many people showed up." GMV per viewer answers "how much did each of those people account for in sales." A small room can win the ratio. S05 did: 379 people, and only 6,852.76 CNY of GMV. S12 is behind by 0.05 CNY per viewer and brought in 25,728.68 CNY.

**What I would tell the founder.** Do not use peak viewers as the definition of a good stream. S11's giveaway got 9,500 comments and still has the lowest GMV per viewer, 11.88 CNY. GMV per viewer is the better of the two scores, because the store is trying to sell. Do not use it alone either. S05 wins it by a hair and is a small session. A good stream is a high GMV per viewer in a large room. That is S12. Repeat S12, not S05.


## Task 1.2 · Month two

Growth = `gmv_w4 / gmv_w1`. The percent change is that ratio minus 1. Both rankings pick the same two SKUs. The weekly file is store-wide GMV, not livestream GMV.


In [4]:
w = pd.read_csv("livestream_weekly_sales.csv")
w["growth"] = w["gmv_w4"] / w["gmv_w1"]
w["growth_pct"] = (w["growth"] - 1) * 100
week4_total = w["gmv_w4"].sum()
w["week4_share"] = w["gmv_w4"] / week4_total

ranked_sku = w.sort_values("growth", ascending=False).reset_index(drop=True)
print(f"Week 4 store GMV: {week4_total:,.2f} CNY")
print(ranked_sku[["sku", "product_name", "price", "gmv_w1", "gmv_w2", "gmv_w3", "gmv_w4",
                  "growth", "growth_pct", "week4_share"]].round(3).to_string(index=False))

top2 = ranked_sku.head(2)
for _, row in top2.iterrows():
    units_w4 = row["gmv_w4"] / row["price"]
    print(f"{row['sku']} {row['product_name']}: growth {row['growth']:.3f} "
          f"({row['growth_pct']:.1f}%), week-4 units at list price about {units_w4:.0f}")


Week 4 store GMV: 71,616.61 CNY
sku       product_name  price   gmv_w1   gmv_w2   gmv_w3   gmv_w4  growth  growth_pct  week4_share
P02    Cloud Cream 50g    199 15418.24 18925.89 22730.17 30296.71   1.965      96.499        0.423
P03   Mist Toner 150ml     89  2466.52  2924.40  3451.34  4544.28   1.842      84.239        0.063
P05        Lip Tint 04     69  1601.11  1991.49  2182.68  2866.81   1.791      79.051        0.040
P06       Eye Gel 15ml    129  4207.01  4738.52  6169.41  7277.29   1.730      72.980        0.102
P04     Clay Mask 100g     79  2817.02  3568.61  3807.01  4366.44   1.550      55.002        0.061
P01   Hydra Serum 30ml    169  5922.82  7268.33  8198.00  9072.81   1.532      53.184        0.127
P07     Sun Milk SPF50     99  3815.54  4198.98  4431.21  5775.08   1.514      51.357        0.081
P08 Cleansing Balm 90g    109  5103.71  5512.18  6316.92  7417.19   1.453      45.329        0.104
P02 Cloud Cream 50g: growth 1.965 (96.5%), week-4 units at list price about 1

**The two fastest.** Cloud Cream 50g (P02), then Mist Toner 150ml (P03).

Cloud Cream went from 15,418.24 CNY in week 1 to 30,296.71 CNY in week 4. Growth is 1.965, a 96.5% increase, and it rose every week. In week 4 it was 42% of store GMV. At the 199 CNY list price, week 4 is about 152 units.

**Action for Cloud Cream.** Before month two, buy enough to hold the week-4 pace: about 150 units a week, so about 600 units for four weeks. Put it first in every livestream. It is the fastest grower and the biggest seller.

Mist Toner went from 2,466.52 CNY to 4,544.28 CNY. Growth is 1.842, an 84.2% increase, and it also rose every week. It is still small: 6.3% of week-4 sales. At 89 CNY, week 4 is about 51 units.

**Action for Mist Toner.** Give it one fixed pitch in each livestream, as the cheaper step in front of Cloud Cream, and bring in about 50 units a week. Do not buy it as if it were the hero SKU. In month two, check whether weekly GMV stays near 4,544 CNY.


## Task 1.3 · Did the streams pay for themselves?

Cost = 800 CNY × total hours. Hours = total minutes / 60, on the 12 sessions after the duplicate is removed. The comparison is with stream GMV (the sum of the session file), not with the weekly store file.


In [5]:
hours = df["duration_min"].sum() / 60
cost = 800 * hours
stream_gmv = df["gmv"].sum()
store_gmv = w[["gmv_w1", "gmv_w2", "gmv_w3", "gmv_w4"]].to_numpy().sum()

print(f"Minutes: {df['duration_min'].sum():.0f}")
print(f"Hours: {hours:.1f}")
print(f"Streaming cost at 800 CNY/hour: {cost:,.0f} CNY")
print(f"Stream GMV: {stream_gmv:,.2f} CNY")
print(f"Store GMV, four weeks: {store_gmv:,.2f} CNY")
print(f"Streams as a share of store GMV: {stream_gmv / store_gmv:.0%}")
print(f"Stream GMV / streaming cost: {stream_gmv / cost:.1f}")
print(f"Break-even margin: {cost / stream_gmv:.2%}")

print()
print("What the store keeps if the margin is...")
for margin in (0.30, 0.15, 0.1006, 0.08):
    kept = margin * stream_gmv
    covers = "covers the 18,400" if kept >= cost else "does not cover the 18,400"
    print(f"  {margin:.2%}  keeps {kept:,.0f} CNY  {covers}")

print()
print(f"If the hourly cost were 1,600 CNY instead of 800, the bill would be {1600 * hours:,.0f} CNY.")
print(f"A 30% margin ({0.30 * stream_gmv:,.0f} CNY) would still cover that.")
print(f"An 8% margin ({0.08 * stream_gmv:,.0f} CNY) fails even at the original 800.")


Minutes: 1380
Hours: 23.0
Streaming cost at 800 CNY/hour: 18,400 CNY
Stream GMV: 182,907.21 CNY
Store GMV, four weeks: 219,383.72 CNY
Streams as a share of store GMV: 83%
Stream GMV / streaming cost: 9.9
Break-even margin: 10.06%

What the store keeps if the margin is...
  30.00%  keeps 54,872 CNY  covers the 18,400
  15.00%  keeps 27,436 CNY  covers the 18,400
  10.06%  keeps 18,400 CNY  covers the 18,400
  8.00%  keeps 14,633 CNY  does not cover the 18,400

If the hourly cost were 1,600 CNY instead of 800, the bill would be 36,800 CNY.
A 30% margin (54,872 CNY) would still cover that.
An 8% margin (14,633 CNY) fails even at the original 800.


**Conclusion.** The 12 sessions are 23.0 hours. The bill is 18,400 CNY. Stream GMV is 182,907.21 CNY, about 9.9 times the bill, and about 83% of the month's store GMV (219,383.72 CNY).

That gap is not profit. GMV still has to pay for the product and for platform fees. The 800 CNY only pays the host, the room, and the samples.

The result turns on the margin, meaning the share of GMV the store keeps.

- At 30%, the streams contribute about 54,872 CNY. That covers 18,400.
- The line is 10.06% (18,400 / 182,907.21).
- At 8%, they contribute about 14,633 CNY. That does not cover 18,400.

So the streams paid for themselves if the store keeps more than about 10% of stream GMV, and they did not if the margin is thinner. This is the assumption the conclusion stands or falls with. Doubling the hourly cost to 1,600 CNY raises the bill to 36,800 CNY, and a 30% margin still covers it. An 8% margin fails at the original 800. The margin matters more than a small mistake in the hourly rate.


## Part 2 · BakeBox, August 2026

Same steps as the Week 3 notebook. The export is one row per order line. A basket is the set of products in one order. There are 2,800 orders and two channels.


In [6]:
NAME = {
    "BB01": "bread_flour", "BB02": "instant_yeast", "BB03": "cake_flour",
    "BB04": "matcha_powder", "BB05": "cocoa_powder", "BB06": "sugar",
    "BB07": "butter", "BB08": "bread_improver", "BB09": "cake_mold",
    "BB10": "bamboo_whisk", "BB11": "piping_bag", "BB12": "nozzle_set",
    "BB13": "spatula_set", "BB14": "mixing_bowl", "BB15": "kitchen_scale",
    "BB16": "oven_thermometer", "BB17": "baking_tray", "BB18": "parchment_roll",
    "BB19": "cupcake_cases", "BB20": "doilies", "BB21": "gift_boxes",
    "BB22": "sprinkle_mix", "BB23": "food_coloring", "BB24": "cake_toppers",
    "BB25": "apron",
}

raw = pd.read_csv("order_baskets.csv")
print("order lines:", len(raw))
raw["item"] = raw["sku"].map(NAME)

orders = raw.groupby("order_id").agg(
    channel=("channel", "first"),
    items=("item", lambda s: sorted(set(s))),
)
print("orders:", len(orders))
print(orders["channel"].value_counts().to_string())


order lines: 7322
orders: 2800
channel
storefront    1984
livestream     816


In [7]:
def rule_text(r):
    left = " + ".join(sorted(r["antecedents"]))
    right = " + ".join(sorted(r["consequents"]))
    return left + " -> " + right

def rank_rules(rules, price):
    view = rules[["rule", "support", "confidence", "lift"]].copy()
    view["lift_shown"] = view["lift"].round(3)
    view["left_items"] = rules["antecedents"].apply(len)
    view["left_price"] = rules["antecedents"].apply(lambda a: float(price[list(a)].sum()))
    view = view.sort_values(
        ["lift_shown", "left_items", "left_price"],
        ascending=[False, True, True],
    )
    return view[["rule", "support", "confidence", "lift"]].reset_index(drop=True)

def mine(sub, min_support=0.05):
    te = TransactionEncoder()
    onehot = pd.DataFrame(
        te.fit(sub["items"]).transform(sub["items"]),
        columns=te.columns_,
        index=sub.index,
    )
    frequent = apriori(onehot, min_support=min_support, use_colnames=True)
    rules = association_rules(frequent, metric="confidence", min_threshold=0.0)
    rules["rule"] = rules.apply(rule_text, axis=1)
    return onehot, frequent, rules

price = raw.groupby("item")["unit_price"].first()
onehot, frequent, rules = mine(orders)
print(f"{len(frequent)} frequent itemsets at min_support 0.05")
print(f"{len(rules)} rules")
ranked = rank_rules(rules, price)
print(ranked.head(12).round(3).to_string(index=False))


39 frequent itemsets at min_support 0.05
40 rules
                              rule  support  confidence   lift
apron -> mixing_bowl + spatula_set    0.079       0.804 10.091
mixing_bowl + spatula_set -> apron    0.079       0.991 10.091
mixing_bowl -> apron + spatula_set    0.079       0.759  9.622
apron + spatula_set -> mixing_bowl    0.079       1.000  9.622
spatula_set -> apron + mixing_bowl    0.079       0.759  9.536
apron + mixing_bowl -> spatula_set    0.079       0.991  9.536
              apron -> mixing_bowl    0.080       0.811  7.803
              mixing_bowl -> apron    0.080       0.766  7.803
     bamboo_whisk -> matcha_powder    0.089       0.852  7.798
     matcha_powder -> bamboo_whisk    0.089       0.810  7.798
              spatula_set -> apron    0.079       0.759  7.733
              apron -> spatula_set    0.079       0.804  7.733


## Task 2.1 · The strongest rules

Support is how often the two sides show up together. Confidence is how often the right-hand side is bought once the left-hand side is already in the basket. Lift compares that with chance. Lift near 1 means no real link.


In [8]:
top3 = ranked.head(3)
print("Three rules with the highest lift")
print(top3.round(3).to_string(index=False))

chosen = ranked.iloc[1]
print()
print("Rule I would use:")
print(f"  {chosen['rule']}")
print(f"  support {chosen['support']:.3f}, confidence {chosen['confidence']:.3f}, lift {chosen['lift']:.3f}")
print(f"  orders together: about {chosen['support'] * len(orders):.0f} of {len(orders)}")


Three rules with the highest lift
                              rule  support  confidence   lift
apron -> mixing_bowl + spatula_set    0.079       0.804 10.091
mixing_bowl + spatula_set -> apron    0.079       0.991 10.091
mixing_bowl -> apron + spatula_set    0.079       0.759  9.622

Rule I would use:
  mixing_bowl + spatula_set -> apron
  support 0.079, confidence 0.991, lift 10.091
  orders together: about 221 of 2800


The top three rules are one kit: the apron, the mixing bowl, and the spatula set.

The rule to act on is **mixing bowl + spatula set → apron**. Support 0.079, confidence 0.991, lift 10.091. The three items are together in about 8% of orders. When the bowl and the spatula are already chosen, the apron is bought 99.1% of the time, about 10 times as often as chance.

**What the merchant should do.** Sell the three as one bundle, on the product page and in the live show. If a cart already has two of them, offer the third. Do not cut the price of one piece in a way that splits the set. Customers are already buying them together.


## Task 2.2 · The bestseller that goes with nothing

Doilies have the highest support in the store. Most doily pairs sit below the 0.05 line, so Apriori will not list them. I compute the lift by hand, the same way as section 7 of the Week 3 notebook: confidence divided by the other item's base rate.


In [9]:
doily_support = onehot["doilies"].mean()
print(f"doilies support: {doily_support:.3f}  ({int(onehot['doilies'].sum())} of {len(onehot)} orders)")

rows = []
for col in onehot.columns:
    if col == "doilies":
        continue
    both = int((onehot["doilies"] & onehot[col]).sum())
    lift = (both / onehot["doilies"].sum()) / onehot[col].mean()
    rows.append((col, both, round(float(lift), 3)))

decoy = pd.DataFrame(rows, columns=["item", "orders_together", "lift_with_doilies"])
decoy = decoy.sort_values("lift_with_doilies").reset_index(drop=True)
print(f"lift range: {decoy['lift_with_doilies'].min():.3f} to {decoy['lift_with_doilies'].max():.3f}")
print(decoy.to_string(index=False))

doily_rules = rules[rules["rule"].str.contains("doilies")]
print(f"rules that contain doilies at min_support 0.05: {len(doily_rules)}")
print(f"largest doily pair, in orders: {int(decoy['orders_together'].max())} (the 5% line is {0.05 * len(orders):.0f})")


doilies support: 0.280  (784 of 2800 orders)
lift range: 0.804 to 1.250
            item  orders_together  lift_with_doilies
           sugar               45              0.804
     baking_tray               40              0.893
       cake_mold              112              0.952
   matcha_powder               82              0.957
    bamboo_whisk               79              0.970
   food_coloring               44              0.982
   kitchen_scale               44              0.982
  bread_improver               78              0.984
    cocoa_powder               47              0.987
     bread_flour              116              0.998
           apron               77              1.000
  parchment_roll              133              1.002
   instant_yeast              117              1.004
    sprinkle_mix               45              1.004
    cake_toppers               45              1.004
      cake_flour              119              1.010
      nozzle_set           

Doilies are in 784 of 2,800 orders, support 0.280. That is the most common product. The lift with every other product runs from 0.804 to 1.250. Nothing is far from 1. At min_support 0.05, the mining returns no doilies rule. The biggest pair is still under 140 orders.

High support means the item is common. It does not mean the item predicts anything else. A cross-sell anchor needs confidence and a lift clearly above 1. Doilies have neither. A "people who bought doilies also bought this" box would just repeat the store's ordinary popular items.

**Keep or delist.** Do not use association for this decision. Use the item's own profit: how many are sold, the margin on each one, and what they cost to store and pack. Doilies are in 28% of orders, so they are a volume item. Keep them if that volume is profitable. Drop them if they lose money, even though they are the bestseller by count.


## Task 2.3 · One store, two channels

Same mining, same 0.05 support, once on livestream orders and once on storefront orders. The bar is 5% of that channel, not 5% of the whole store.


In [10]:
live = orders[orders["channel"] == "livestream"]
store = orders[orders["channel"] == "storefront"]
_, frequent_live, rules_live = mine(live)
_, frequent_store, rules_store = mine(store)
print(f"{len(live)} livestream orders, {len(frequent_live)} itemsets, {len(rules_live)} rules")
print(f"{len(store)} storefront orders, {len(frequent_store)} itemsets, {len(rules_store)} rules")

def show_rule(rules, text):
    hit = rules[rules["rule"] == text]
    if hit.empty:
        print(f"{text}: ABSENT")
    else:
        print(hit[["rule", "support", "confidence", "lift"]].round(3).to_string(index=False))

print()
print("Livestream")
show_rule(rules_live, "spatula_set -> apron")
print("Storefront")
show_rule(rules_store, "spatula_set -> apron")

print()
print("Two storefront rules the bundle page can use")
store_ranked = rank_rules(rules_store, price)
print(store_ranked.head(4).round(3).to_string(index=False))


816 livestream orders, 37 itemsets, 72 rules
1984 storefront orders, 32 itemsets, 32 rules

Livestream
                rule  support  confidence  lift
spatula_set -> apron    0.271         1.0 3.692
Storefront
spatula_set -> apron: ABSENT

Two storefront rules the bundle page can use
                         rule  support  confidence  lift
     piping_bag -> nozzle_set    0.109       1.000 9.185
     nozzle_set -> piping_bag    0.109       1.000 9.185
bamboo_whisk -> matcha_powder    0.100       1.000 8.371
matcha_powder -> bamboo_whisk    0.100       0.835 8.371


**The rule that is live-only.** Spatula set → apron.

On the livestream (816 orders): support 0.271, confidence 1.000, lift 3.692. Every live order with the spatula set also has the apron. The pair is in 27.1% of live orders.

On the storefront (1,984 orders): the pair does not reach support 0.05. The rule is absent.

**Why.** The host pushes this set on the show. A confidence of 1.0 is the script. On the storefront nobody makes that pitch, so the pair is too rare for the same 5% line. The blended file still shows the apron / bowl / spatula kit at a lift of about 10, because the live orders are mixed in. Reading only the blended list makes the kit look like a habit of the whole store. It is not.

**One rule set per channel, not one blended set.**

- Storefront bundle page: use the storefront rules. The strong ones there are baking pairs, for example piping bag → nozzle set (support 0.109, confidence 1.000, lift 9.185) and bamboo whisk → matcha powder (support 0.100, confidence 1.000, lift 8.371). Do not put the apron kit in the hero slot on that page.
- Live show: keep offering the spatula set with the apron and the mixing bowl. In that room the confidence is 1.0. The host can also mention the baking pairs, but the script should follow the live rules, not the storefront page.


## Figures the report uses

This cell reprints the numbers cited in the PDF, so the report and the notebook stay tied together.


In [11]:
print("1.1 S05", f"{df.loc[df.session_id=='S05','gmv_per_viewer'].iloc[0]:.2f}",
      "S12 viewers", int(df.loc[df.session_id=='S12','viewers_peak'].iloc[0]),
      "S12 ratio", f"{df.loc[df.session_id=='S12','gmv_per_viewer'].iloc[0]:.2f}")
print("1.2 top two", list(top2["product_name"]), list(top2["growth"].round(3)), list(top2["growth_pct"].round(1)))
print("1.3 cost", f"{cost:.0f}", "stream gmv", f"{stream_gmv:.2f}", "margin", f"{cost/stream_gmv:.4f}")
print("2.1", top3.round(3)[["rule","support","confidence","lift"]].values.tolist())
print("2.2 doilies", f"{doily_support:.3f}", "lifts",
      f"{decoy['lift_with_doilies'].min():.3f}", f"{decoy['lift_with_doilies'].max():.3f}")
print("2.3 live rules", len(rules_live), "store rules", len(rules_store))


1.1 S05 18.08 S12 viewers 1427 S12 ratio 18.03
1.2 top two ['Cloud Cream 50g', 'Mist Toner 150ml'] [1.965, 1.842] [96.5, 84.2]
1.3 cost 18400 stream gmv 182907.21 margin 0.1006
2.1 [['apron -> mixing_bowl + spatula_set', 0.079, 0.804, 10.091], ['mixing_bowl + spatula_set -> apron', 0.079, 0.991, 10.091], ['mixing_bowl -> apron + spatula_set', 0.079, 0.759, 9.622]]
2.2 doilies 0.280 lifts 0.804 1.250
2.3 live rules 72 store rules 32
